In [1]:
import torch
from torchvision import transforms
from torchvision.models import efficientnet_v2_l, EfficientNet_V2_L_Weights
from PIL import Image

In [3]:

# Path to image and model
image_path = 'img.jpg'
model_path = 'best_soil_model.pth'

# Define the class index mapping (adjust if needed)
class_idx = {0: 'dry', 1: 'high', 2: 'low', 3: 'medium'}  # Replace by printing train_dataset.class_to_idx for your case

# Define preprocessing (must match validation/test)
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = efficientnet_v2_l(weights=EfficientNet_V2_L_Weights.DEFAULT)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 4)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
model.to(device)

# Load and preprocess the image
img = Image.open(image_path).convert('RGB')
img_tensor = preprocess(img).unsqueeze(0).to(device)

# Predict
with torch.no_grad():
    outputs = model(img_tensor)
    _, predicted = outputs.max(1)
    predicted_class = class_idx[predicted.item()]  # Map label index to class name

print(f"Predicted class: {predicted_class}")


Predicted class: high
